# RAG Minecraft

## Configuration Gemini API

In [30]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time


In [31]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

Congig sources

In [32]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [33]:
scraper = cloudscraper.create_scraper()

In [34]:
def write_csv(file_name, paragraphs):
    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

In [35]:
def write_csv(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

Wikipedia

In [36]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    texts = []
    capture = False

    for tag in soup.find_all(["h2", "p"]):

        t = tag.get_text(" ", strip=True)

        if tag.name == "h2" and "Références" in t:
            break

        if tag.name == "h2":
            capture = True
            continue

        if capture and t:
            texts.append(t)

    write_csv("files/wikipedia.csv", texts)

    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": f"wikipedia:{title}"}
    )

Fandom

In [37]:
def scrape_fandom(url: str):

    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return None

    texts = []

    for tag in content.find_all(["p", "h2", "h3", "li"]):

        t = tag.get_text(" ", strip=True)

        if not t:
            continue

        if len(t) < 20:
            continue

        if "Voir aussi" in t or "Notes et références" in t:
            break

        texts.append(t)

    paragraphs = [p.strip() for p in texts if p.strip()]
    page_name = url.split("/wiki/")[-1]
    page_name = unquote(page_name)
    file_name = f"files/{page_name}.csv"

    write_csv(file_name, paragraphs)


    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": url}
    )

Build dataset

In [38]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.append(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
for url in FANDOM_URLS:
    try:
        doc = scrape_fandom(url)
        if doc:
            docs.append(doc)
            print("FANDOM OK:", url)
    except Exception as e:
        print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))

WIKI OK: Minecraft
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Survie
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Hardcore
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Fabrication
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cuisson
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Alchimie
FANDOM OK: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures

TOTAL DOCS: 12


Chunking

In [39]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

print("CHUNKS:", len(split_docs))

CHUNKS: 105


Enrchich

In [40]:
def enrich_with_source(docs):
    enriched = []

    for d in docs:
        src = d.metadata.get("source", "unknown")

        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n\n"
            f"{d.page_content}"
        )

        enriched.append(d)

    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [41]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

#### Store data using chroma

In [42]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

dossier encore utilisé → restart kernel puis relance


In [43]:
persist_dir = "./chroma_minecraft_db"

# 1. Création + stockage (UNE SEULE FOIS)
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)
batch_size = 10

for i in range(0, len(split_docs), batch_size):
    batch = split_docs[i:i+batch_size]
    vectorstore.add_documents(batch)
    time.sleep(10)

# 2. Reload propre (pour usage RAG)
vectorstore_disk = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)

# 3. Retriever FINAL utilisé par le RAG
retriever = vectorstore_disk.as_retriever(
    search_kwargs={"k": 6}
)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

split_docs: 98
chroma: 98


#### Create a retriever using Chroma

In [ ]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

SOURCE: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
SOURCE_TYPE: FANDOM

Q: Je meurs tout le temps dès la première nuit ! Aidez-moi ! [ ]

Q: Comment changer l'apparence de mon personnage ? [ ]

Q: Le niveau de l'eau peut-il monter ? [ ]

Q: Quelles sont les touches ? [ ]

Q: Comment jeter d'un coup une pile d'objets ? [ ]

Q: Comment sauvegarder/charger ma position en multijoueur ? [ ]

Q: Comment faire pousser des plantes ? Elles ne font que disparaître… [ ]

Q: Pourquoi est-ce


## Generator

#### Initialize Generator

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite")

#### Create prompt templates

In [ ]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='Tu es un assistant expert sur le jeu Minecraft.\nRéponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.\nSi la réponse ne se trouve pas dans le contexte ou si tu n\'es pas sûr, n\'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l\'information n\'est pas dans les documents fournis."\nSois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l\'information.\n\nQuestion: {question}\nContexte: {context}\nRéponse:'


#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

▶ Question posée : Quel est l'ingrédient de base indispensable pour l'alchimie ?


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 9.741746866s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemin

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

▶ Question posée : C'est quoi le Warden dans Minecraft ?
Succès du Niveau 1.


Le Warden est un monstre de Minecraft, parfois confondu avec un boss, qui vit dans une « cité antique » située dans les grottes les plus basses du monde. Apparu en 2022 lors de la mise à jour 1.19 (1.19.0.20 pour la version Bedrock), il est capable d'infliger de très lourds dégâts. À sa mort, il laisse tomber un bloc appelé « catalyseur de sculk ».

[WIKIPEDIA: wikipedia:Minecraft]

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

▶ Question posée : Comment survivre dans le Nether dans Minecraft ?
Succès du Niveau 1.


Pour survivre dans le Nether, il est conseillé de construire un abri sûr (de préférence en pierre taillée ou en roche) autour de votre portail afin de résister aux boules de feu des Ghasts. Vous devez emporter un briquet pour pouvoir rallumer votre portail si nécessaire, et ne jamais poser de lit car il explose si vous tentez d'y dormir. Il est également recommandé de marquer votre chemin avec des torches ou des blocs visibles pour éviter de vous perdre. Contre les Ghasts, vous pouvez essayer de leur renvoyer leurs projectiles en faisant un clic gauche dessus. Enfin, pour explorer, munissez-vous d'une pioche et d'une épée en fer, d'un bouclier, de torches et de beaucoup de nourriture. [SOURCE_TYPE: FANDOM]

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

▶ Question posée : C'est quoi l'alchimie dans Minecraft?


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 45.360824501s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
Markdown(ask_with_active_retrieval("Quels sont les biomes dans Minecraft?"))

▶ Question posée : Quels sont les biomes dans Minecraft?
Information introuvable. Activation de l'Active Retrieval...
Nouvelle requête optimisée par l'IA : [{'type': 'text', 'text': 'liste biomes Minecraft', 'extras': {'signature': 'EoEOCv4NAQw51scgxc/eXMuSlcUETpbU6r5mKG8a7ZK1K2j7+SBKb/7Qqx7PI5rKPYNUkt3IvQaXulvHbI0NeBobE22qedYO5di79zHen3BH9RiWEWOfVtuPDFqmD27q6Opvk4OmaZ6r5SaOjfYbR+hvfdsuEMCNfBBVjBfMHg2HJvdwWoZinr2QmVuoyk8HY0wWSP3ZDsehtthXD596qGCZ8IV0kDgMWMiisOJRPHH8QpQCPixZsNd4ol7MPyZ+F2IxwLdCXTtcxh0LLxvtIJQIX9bBJ3Ecg/4XWom6FrZoe9i6t/tMJcotkLIwUZsmd06S10wYmEbsBiLY7Vc6McXuAc8QS2MR529RavEIndjfmwZb7+nGta0o5RSc7Q7F5zbEqqx8cdAo723KZriShgGLQxSMqO9y0jLk+wSdLUYGzSANG/fCmxIfmjQY9yCqHugbeZykWE71pIgFMi4FxreCf/Vh+kXzdWDtYOElIfkOoIfojAH2W9shC3Dx2HlyAsSMxNFMPhfucjDNPHLC0Gji4xm9t6Uq/Hh/y1HsO2bmKLC7nw4n9KX7FxZer3F5S1OtX+kqvsbfRNuacVN6n+KSHc7ikGRv00KtRURfHgJ/xZd/EGQTo/Z8EFlsnpHGH6vHb33r0Vx4za0cQt+K/V1uZR0ot0xSfD11s2rYKLnpv31APFqjASmehhjQOmZxlaIaHFYwFmpXpAUwg9XaVQQux0aY+WXO3APw3BEKHWqK1HcIarv5kNCdoH+C8BT

*(Auto-correction RRR : Recherche élargie avec les mots-clés '[{'type': 'text', 'text': 'liste biomes Minecraft', 'extras': {'signature': 'EoEOCv4NAQw51scgxc/eXMuSlcUETpbU6r5mKG8a7ZK1K2j7+SBKb/7Qqx7PI5rKPYNUkt3IvQaXulvHbI0NeBobE22qedYO5di79zHen3BH9RiWEWOfVtuPDFqmD27q6Opvk4OmaZ6r5SaOjfYbR+hvfdsuEMCNfBBVjBfMHg2HJvdwWoZinr2QmVuoyk8HY0wWSP3ZDsehtthXD596qGCZ8IV0kDgMWMiisOJRPHH8QpQCPixZsNd4ol7MPyZ+F2IxwLdCXTtcxh0LLxvtIJQIX9bBJ3Ecg/4XWom6FrZoe9i6t/tMJcotkLIwUZsmd06S10wYmEbsBiLY7Vc6McXuAc8QS2MR529RavEIndjfmwZb7+nGta0o5RSc7Q7F5zbEqqx8cdAo723KZriShgGLQxSMqO9y0jLk+wSdLUYGzSANG/fCmxIfmjQY9yCqHugbeZykWE71pIgFMi4FxreCf/Vh+kXzdWDtYOElIfkOoIfojAH2W9shC3Dx2HlyAsSMxNFMPhfucjDNPHLC0Gji4xm9t6Uq/Hh/y1HsO2bmKLC7nw4n9KX7FxZer3F5S1OtX+kqvsbfRNuacVN6n+KSHc7ikGRv00KtRURfHgJ/xZd/EGQTo/Z8EFlsnpHGH6vHb33r0Vx4za0cQt+K/V1uZR0ot0xSfD11s2rYKLnpv31APFqjASmehhjQOmZxlaIaHFYwFmpXpAUwg9XaVQQux0aY+WXO3APw3BEKHWqK1HcIarv5kNCdoH+C8BThpYmZRQOTGhpNbt+EgkZD6cznBT1SVejfoAWQrfCni9qO7rSR9PSJi5j31RjXwRp0qcbjmnmDQl7hKVEXoUq0POG/cRWfv1jTRO0GQUCB1Hra13BwiTlbpTRyWq9RBy8h+upaJJLYOsVuUbFLcBOrQ68V/chER3xA1EgrfyZCE6Q5Hca6nOvqAaF4Jak7WCWyYrSn5v7dUf8z64GOdJwY2v43DK6MPXatw60NiVsHAGYSZ28MHbTjIJoN+beMPSn6Ys2DZ6eWz/wGXSlBfPUxSxPx3qbxvvNh1NCodXLbPPJptEgyhbmudteUm3eEQd8JIPgfFeQJpfbj6UbnDWWIQ75tfhn8i3HMpfGxzHfLH30b7FqMLMgAJbnAPvOIX4feo2KEUHOsj+u9DKHk4tOPramP0BdonYtuY8RxxZIaCjwo7t9hM4+JMCIhqgvH6wYoK6SokTxR18B9IKuLl0YiZqR9dQo25shMfdREIKCb+LGI98kegV+YKOJb418tLsn/tr+73kZGTbixcNb0+eNxGrXA39lVMfN74XfIHksRpXwkKqplDgkgxjr0fId5iS8NZuw1zAaQvmavJPKsBI1LV+EhnmjrOotJd97A38byk5anNGHeo/lkzEIpDqjbg5w4ZM5+Z4CkQsZvu21M1spvBseOuX9FX60Iv3TazU0BzssfWimKAW1HZTTtwNabeUBr4YmW4wBeuin6E1Xfnj0R4Loy2/cZznzordHtC4Zo0clbKbB/5MzIZYwbCJpWe/TsY7oMhSchWQKXmaFB3KWPcNgIYbgGBpj8d9dydDXrcleGnVoP/V7Ii7FA8V0oigkJ0tN1/bIDPw1Gk3Z/glOy0PJ3MotbVU9UXVKFw+ayanEVzXAqqz/98Af2/FCuf8wOoqk/iEMu+88PSXBPCafDMkv31WhvbyrLF2Tw7xJTEuX8wdnKZWvn7bRWxY7XWh5o1PgLpX5vrqLIt2DGG2A1uV0qzH48owztnhkCW6Q1FqjMOm2ol7SysgJvWuvDp3U/b9dUIpf+pc+WwFKFumR+xUZmGorhpAkdWluVFWXuBRh+hqX0sL1+KwcAC4rGVCYMYCPvzCMu+JiK83ye7cPnw4Og08QgspsFYKReVZcp0ffkY/6X/VroTcsinHxQJGW5EzJGMDjTsqTpW8/n2IpIAv+ooSFq9aE/IBcsZGoSJHuJERWM16Q84hHjWwmQk9yXisjG5EDB6DeDHDj46x6FVAIE+6sTR3ti7skeX0EMGRHM+78TMDZNjD4mu7E07rvyVVYStpiTGkTg5r08JYxUHHY8bFbWZiPfSfO7YCp9I46wngZcyDcqrhl7sBqR4M8KnrXrLLKXVDaMHucXSlN+m6yN9ff8TgaSO9fmp7mkam9Nfvv8C2wsDXlsDMNMRl/X7hCGa675vm4Zr07KK7XV7MT5nd38sFdWH+NABZpEGEwUqgXvZF/dZk254FOISM1qKrAqhSiOelshyHyS87xuaGvf5I9uIIHSZh6P7juPpXrLU3422o3u7PM1MBtOojm2KHp8tlm5kIe6hnn9gKnG51SJO7i7/UNPkvCtlN05pwNyZNRTBIrP6p8QTKjfuNo+cXp018i+bQ5FuMyR6TSdQum5AJk='}}]')*

D'après les documents fournis, le monde de Minecraft est composé de différents biomes tels que la forêt, la plaine, le désert, la jungle, la toundra, la taïga, le marais et la savane [WIKIPEDIA: wikipedia:Minecraft]. Les documents mentionnent également l'existence d'un biome enneigé [FANDOM: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures].

In [ ]:
Markdown(rag_chain.invoke("Est ce que je peux miner du diamant avec une pioche en pierre dans Minecraft?"))

Non, vous ne pouvez pas miner du diamant avec une pioche en pierre pour récupérer sa ressource. Si vous n'avez pas de pioche en fer (ou en diamant), le diamant ne lâchera pas son précieux contenu et sera tout simplement détruit. [SOURCE_TYPE: FANDOM]

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft?"))

▶ Question posée : Parle moi des boss dans Minecraft?
Succès du Niveau 1.


Les boss dans Minecraft sont des créatures hostiles spéciales dotées d'attaques et de mouvements complexes, ainsi que de très nombreux points de vie. Ils sont généralement immunisés contre la plupart des effets de potions. De plus, ils sont capables de voir les créatures affectées par l'effet Invisibilité. Il existe au total trois boss dans le jeu.

[FANDOM: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures]

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft?"))

▶ Question posée : Il y a combien de dimensions dans Minecraft?
Succès du Niveau 1.


Chaque monde de Minecraft comprend le monde principal ainsi que deux autres dimensions : le Nether et l'End, soit un total de trois dimensions. [WIKIPEDIA: wikipedia:Minecraft]

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)